## 1. Train Swin Transformer - Full Fine-Tune


In [11]:
# Swin Transformer - Full Fine-Tune

import os
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import swin_t

# DEVICE CONFIGURATION
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# PATH CONFIG
base_dir = r"D:\alzheimer detection.v1i.folder\processed_dataset"
save_dir = r"D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model"
os.makedirs(save_dir, exist_ok=True)
model_pkl_path = os.path.join(save_dir, "swin_finetune_streamlit.pkl")

# HYPERPARAMETERS
img_size = 224
batch_size = 32
num_classes = 4
epochs = 20
lr = 1e-4
patience = 3
num_workers = 2

# DATA TRANSFORMS
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

train_dataset = datasets.ImageFolder(os.path.join(base_dir, "train"), transform=transform)
val_dataset   = datasets.ImageFolder(os.path.join(base_dir, "test"), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

class_names = train_dataset.classes
print("Classes:", class_names)

# MODEL INITIALIZATION
swin_model = swin_t(weights="IMAGENET1K_V1")
swin_model.head = nn.Linear(swin_model.head.in_features, num_classes)
swin_model = swin_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(swin_model.parameters(), lr=lr)
scaler = torch.cuda.amp.GradScaler()

# TRAINING FUNCTION
# TRAINING FUNCTION
def train_swin(epochs, patience):
    best_val_acc = 0
    trigger_times = 0

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    epochs_trained = 0  # inisialisasi

    for epoch in range(epochs):
        epochs_trained = epoch + 1  # update setiap iterasi

        # TRAIN
        swin_model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epochs_trained}/{epochs} [Train]"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = swin_model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            _, preds = outputs.max(1)
            total += labels.size(0)
            correct += preds.eq(labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # VALIDATION
        swin_model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                with torch.cuda.amp.autocast():
                    outputs = swin_model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = outputs.max(1)
                val_total += labels.size(0)
                val_correct += preds.eq(labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total

        # RECORD METRICS
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"[Epoch {epochs_trained}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # BEST MODEL CHECK
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f"Early stopping triggered at epoch {epochs_trained}")
                break

    # SAVE SINGLE PKL
    artifact = {
        "model_state_dict": swin_model.state_dict(),
        "architecture": "swin_t (Full Fine-Tune)",
        "num_classes": num_classes,
        "class_names": class_names,
        "img_size": img_size,
        "normalization": {"mean": [0.5,0.5,0.5], "std": [0.5,0.5,0.5]},
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accs": train_accs,
        "val_accs": val_accs,
        "best_val_acc": best_val_acc,
        "epochs_trained": epochs_trained
    }

    with open(model_pkl_path, "wb") as f:
        pickle.dump(artifact, f)

    print(f"\n[SUCCESS] Training selesai! Model + metrics tersimpan di: {model_pkl_path}")
    return model_pkl_path


train_swin(epochs=epochs, patience=patience)

DEVICE: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


C:\Users\acer\AppData\Local\Temp\ipykernel_39412\3643058269.py:55: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/20 [Train]:   0%|          | 0/184 [00:00<?, ?it/s]C:\Users\acer\AppData\Local\Temp\ipykernel_39412\3643058269.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/20 [Train]: 100%|██████████| 184/184 [03:29<00:00,  1.14s/it]
C:\Users\acer\AppData\Local\Temp\ipykernel_39412\3643058269.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Epoch 1] Train Acc: 0.6704 | Val Acc: 0.7431 | Train Loss: 0.6968 | Val Loss: 0.5575


Epoch 2/20 [Train]: 100%|██████████| 184/184 [03:28<00:00,  1.13s/it]


[Epoch 2] Train Acc: 0.7537 | Val Acc: 0.7666 | Train Loss: 0.5220 | Val Loss: 0.4969


...

Epoch 13/20 [Train]: 100%|██████████| 184/184 [05:35<00:00,  1.82s/it]


[Epoch 13] Train Acc: 0.9882 | Val Acc: 0.8936 | Train Loss: 0.0336 | Val Loss: 0.4062


Epoch 14/20 [Train]: 100%|██████████| 184/184 [05:17<00:00,  1.72s/it]


[Epoch 14] Train Acc: 0.9841 | Val Acc: 0.9381 | Train Loss: 0.0456 | Val Loss: 0.2267
Early stopping triggered at epoch 14

[SUCCESS] Training selesai! Model + metrics tersimpan di: D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\swin_finetune_streamlit.pkl


'D:\\alzheimer detection.v1i.folder\\Dashboard\\src\\Transformer\\model\\swin_finetune_streamlit.pkl'

## 2. Train EfficientFormer

In [ ]:
# EfficientFormer L3 - Full Fine-Tune Training Script

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from timm import create_model
import joblib
from tqdm import tqdm

# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# PATH CONFIG
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"
SAVE_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, "efficientformer_full_finetune_streamlit.pkl")

# CONFIG
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 30
PATIENCE = 5
LR = 3e-5
WEIGHT_DECAY = 0.05

# DATA TRANSFORMS
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(os.path.join(BASE_DIR, "train"), transform=transform)
val_ds   = datasets.ImageFolder(os.path.join(BASE_DIR, "test"), transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

class_names = train_ds.classes
print("Classes:", class_names)


# MODEL: EfficientFormer L3 - FULL FINE-TUNE
model = create_model("efficientformer_l3", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Full fine-tune: semua parameter dilatih
for param in model.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler()  # Mixed precision


# TRAINING LOOP
best_val_acc = 0.0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("Starting FULL fine-tune EfficientFormer L3...")
for epoch in range(EPOCHS):
    # TRAIN
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # VALIDATION
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # Record metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # SAVE BEST (.pkl Streamlit-ready)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "efficientformer_l3 (Full Fine-Tune)",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            },
            "best_val_acc": best_val_acc,
            "epoch": epoch + 1,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> BEST MODEL & METRICS SAVED! Val Acc: {best_val_acc:.4f} -> {MODEL_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(">>> Early stopping triggered!")
            break

print("\n[SUCCESS] Full Fine-Tune EfficientFormer L3 selesai!")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Model & Metrics .pkl siap untuk Streamlit: {MODEL_PATH}")

DEVICE: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


C:\Users\acer\AppData\Local\Temp\ipykernel_39412\3290777499.py:73: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # Mixed precision


Starting FULL fine-tune EfficientFormer L3...


Epoch 1/30 [Train]:   0%|          | 0/184 [00:00<?, ?it/s]C:\Users\acer\AppData\Local\Temp\ipykernel_39412\3290777499.py:96: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Train]: 100%|██████████| 184/184 [03:41<00:00,  1.21s/it]
C:\Users\acer\AppData\Local\Temp\ipykernel_39412\3290777499.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Epoch 1] Train Acc: 0.6711 | Val Acc: 0.7508 | Train Loss: 0.7436 | Val Loss: 0.5448
>>> BEST MODEL & METRICS SAVED! Val Acc: 0.7508 -> D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\efficientformer_full_finetune_streamlit.pkl


...

[Epoch 14] Train Acc: 1.0000 | Val Acc: 0.8941 | Train Loss: 0.0003 | Val Loss: 0.4288


Epoch 15/30 [Train]: 100%|██████████| 184/184 [06:06<00:00,  1.99s/it]


[Epoch 15] Train Acc: 0.9894 | Val Acc: 0.7497 | Train Loss: 0.0330 | Val Loss: 0.9713


Epoch 16/30 [Train]: 100%|██████████| 184/184 [06:06<00:00,  1.99s/it]


[Epoch 16] Train Acc: 0.9807 | Val Acc: 0.8751 | Train Loss: 0.0553 | Val Loss: 0.4504
>>> Early stopping triggered!

[SUCCESS] Full Fine-Tune EfficientFormer L3 selesai!
Best Validation Accuracy: 0.8951
Model & Metrics .pkl siap untuk Streamlit: D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\efficientformer_full_finetune_streamlit.pkl


## 3. ViT Base Patch16/224 - Fine-Tune

In [10]:
# ViT Base Patch16/224 - Fine-Tune HEAD ONLY training script

import os
import joblib
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm

# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# PATH CONFIG
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")  # Sesuaikan folder val jika berbeda

SAVE_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, "vit_base_finetune_head_streamlit.pkl")

# CONFIG
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
NUM_WORKERS = 2
EPOCHS = 20
PATIENCE = 3
LR = 1e-4

# DATA TRANSFORMS
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# DATASET & DATALOADER
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

class_names = train_dataset.classes
print("Classes:", class_names)

# MODEL: ViT Base Patch16/224 - HEAD ONLY FINE-TUNE
model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Freeze backbone, hanya latih head (classifier bernama 'head')
for name, param in model.named_parameters():
    param.requires_grad = "head" in name

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scaler = torch.cuda.amp.GradScaler()  # Mixed precision

# TRAINING LOOP
best_val_acc = 0.0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("Starting training (head only)...")
for epoch in range(EPOCHS):
    # TRAIN
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # VALIDATION
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # Record metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # SAVE BEST (.pkl Streamlit-ready)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "vit_base_patch16_224 (Head Only Fine-Tune)",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            },
            "best_val_acc": best_val_acc,
            "epoch": epoch + 1,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> BEST MODEL & METRICS SAVED! Val Acc: {best_val_acc:.4f} → {MODEL_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(">>> Early stopping triggered!")
            break

print("\n[SUCCESS] ViT Base Patch16/224 (head fine-tune) selesai!")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Model & Metrics .pkl siap untuk Streamlit: {MODEL_PATH}")

Device: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


C:\Users\acer\AppData\Local\Temp\ipykernel_39412\796921513.py:83: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # Mixed precision


Starting training (head only)...


Epoch 1/20 [Train]:   0%|          | 0/184 [00:00<?, ?it/s]C:\Users\acer\AppData\Local\Temp\ipykernel_39412\796921513.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/20 [Train]: 100%|██████████| 184/184 [03:59<00:00,  1.30s/it]
C:\Users\acer\AppData\Local\Temp\ipykernel_39412\796921513.py:126: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Epoch 1] Train Acc: 0.4678 | Val Acc: 0.5904 | Train Loss: 1.2572 | Val Loss: 1.0378
>>> BEST MODEL & METRICS SAVED! Val Acc: 0.5904 → D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\vit_base_finetune_head_streamlit.pkl


...

[Epoch 10] Train Acc: 0.7344 | Val Acc: 0.7307 | Train Loss: 0.6424 | Val Loss: 0.6424


Epoch 11/20 [Train]: 100%|██████████| 184/184 [02:20<00:00,  1.31it/s]


[Epoch 11] Train Acc: 0.7358 | Val Acc: 0.7368 | Train Loss: 0.6351 | Val Loss: 0.6319


Epoch 12/20 [Train]: 100%|██████████| 184/184 [02:23<00:00,  1.28it/s]


[Epoch 12] Train Acc: 0.7435 | Val Acc: 0.7312 | Train Loss: 0.6220 | Val Loss: 0.6264
>>> Early stopping triggered!

[SUCCESS] ViT Base Patch16/224 (head fine-tune) selesai!
Best Validation Accuracy: 0.7378
Model & Metrics .pkl siap untuk Streamlit: D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\vit_base_finetune_head_streamlit.pkl
